
# TP — Séries temporelles batterie et identification d'un modèle électrique avec JAX


Dans ce TP, on étudie plusieurs semaines de données synthétiques correspondant à l'utilisation d'un véhicule électrique équipé d'un pack batterie fictif.

Le pack considéré possède notamment :

- une capacité nominale d'environ **250 Ah**
- une tension nominale d'environ **200 V**
- une énergie nominale d'environ

$$
E_{\mathrm{nom}}
\approx
U_{\mathrm{nom}}\,C_{\max}
=
200\times250
=
50\,000\ \mathrm{Wh}
$$

soit environ 50 kWh

Les données contiennent principalement :

- le courant batterie
- la tension pack
- la température
- le temps

L'objectif du TP est d'abord de se familiariser avec la structure de longues séries temporelles issues d'un usage véhicule, puis de construire progressivement un modèle permettant de reproduire la tension mesurée.

Le TP est organisé en plusieurs étapes :

**Exploration des séries temporelles**
   - visualisation du courant, de la tension et de la température
   - observation de différents scénarios d'usage
   - identification des phases de charge, de décharge et de repos
   - analyse des trous temporels et de l'échantillonnage

**Construction d'un modèle électrique équivalent**
   - relation entre courant, SOC, OCV et tension
   - introduction d'une résistance ohmique et de branches RC
   - discrétisation et simulation du modèle dynamique

**Identification des paramètres du modèle**
   - comparaison tension mesurée / tension simulée
   - utilisation de l'autodifférentiation avec JAX
   - optimisation des paramètres du modèle

**Analyse des résultats**
   - reconstruction du SOC
   - étude des résidus
   - comparaison des paramètres identifiés aux valeurs utilisées pour générer les données
   - discussion sur l'identifiabilité du modèle


# 0. Installation et imports

In [ ]:
# Installation de optax pour l'optimisation en lien avec JAX et de equinox pour
# faire de la programmation orientée objet
!pip -q install equinox optax

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
import equinox as eqx
import optax
import time

from mpl_toolkits.mplot3d import Axes3D # Figure 3D

jax.config.update("jax_enable_x64",True)


In [ ]:
# Charte graphique du TP
COLOR_U="tab:blue"
COLOR_I="tab:orange"
COLOR_T="tab:red"
COLOR_SOC="tab:green"

R0_CMAP="viridis"


In [ ]:
# ============================================================
# PARAMÈTRES MODIFIABLES — valeurs par défaut
# ============================================================

DOWNSAMPLE_STEP=5
MAX_GAP_S=60.0

N_TEST_SPEED=5000

LEARNING_RATE=1e-2
N_EPOCHS_SINGLE=1000
N_EPOCHS_DOUBLE=1000
N_EPOCHS_IDENT=500

DISPLAY_R0_EVERY=2
DISPLAY_R0_UNTIL=30
PRINT_EVERY=100

LAMBDA_CONT=1e3

ZOOM_START=5000
ZOOM_LENGTH=1800

N_IDENT_RUNS=5

# None = séparation automatique à la moitié
SPLIT_INDEX=None

# Borne supérieure du réseau R0
R0_MAX=2.0


# 1. Chargement des données

In [ ]:
#@title Chargement des données

DATA_URL="https://github.com/lilredhood/FakeBatteryData/releases/download/v1.0/synthetic_battery_tp.npz"

!wget -q --show-progress "$DATA_URL" -O synthetic_battery_tp.npz

data=np.load("synthetic_battery_tp.npz")

print(data.files)

In [ ]:
time_s=jnp.array(data["time"])
dt=jnp.array(data["dt"])
I=jnp.array(data["I"])
T=jnp.array(data["T"])
U=jnp.array(data["U"])

SOC_true=jnp.array(data["SOC"])
OCV_true=jnp.array(data["OCV"])
R0_true=jnp.array(data["R0"])

C_MAX_TRUE=float(data["C_MAX"])
SOC0_TRUE=float(data["SOC0"])
TAU0_TRUE=float(data["TAU0"])
TAU_TRUE=jnp.array(data["TAU"])
R_RC_TRUE=jnp.array(data["R_RC"])
OCV_COEF=jnp.array(data["OCV_coefficients"])
VOLTAGE_NOISE_STD=float(data["VOLTAGE_NOISE_STD"])

print("Nombre de points :",len(time_s))
print("Durée :",float(time_s[-1]/3600/24),"jours")



### Données observées et vérité terrain

Dans la première partie du TP, on travaille surtout avec :

$$
I(t),\qquad U(t),\qquad T(t)
$$

Les tableaux `SOC_true`, `R0_true`, `C_MAX_TRUE`, etc. sont conservés uniquement pour pouvoir comparer les résultats identifiés à la **vérité terrain** à la fin.


## Reconstruction des dates


La première mesure correspond à :

$$
\boxed{\text{12 juin 2024 à 11:19:09}}
$$

Le tableau `time_s` contient le temps écoulé en secondes depuis cette date.


In [ ]:
date0=pd.Timestamp(
    "2024-06-12 11:19:09"
)

df=pd.DataFrame({
    "time_s":np.array(time_s),
    "dt_s":np.array(dt),
    "I_A":np.array(I),
    "T_C":np.array(T),
    "U_V":np.array(U)
})

df["date"]=(
    date0
    +pd.to_timedelta(
        df["time_s"],
        unit="s"
    )
)

df["delta_s"]=(
    df["time_s"]
    .diff()
    .fillna(0)
)

print("Début :",df["date"].iloc[0])
print("Fin   :",df["date"].iloc[-1])

df.head()


In [ ]:
df["SOC_true"]=SOC_true

# 2. Exploration globale des séries temporelles


On utilise des **scatter plots** plutôt que des lignes continues.
Pourquoi ?


Convention de signe :

$$
I>0 \Rightarrow \text{décharge}
$$

$$
I<0 \Rightarrow \text{charge}
$$

In [ ]:
# Visualisation des deux mois de données

view=df.iloc[::200]

plt.figure(figsize=(14,4))
plt.scatter(
    view["date"],
    view["I_A"],
    s=2,
    color=COLOR_I
)
plt.ylabel("Courant [A]")
plt.xlabel("Date")
plt.grid()
plt.show()

plt.figure(figsize=(14,4))
plt.scatter(
    view["date"],
    view["U_V"],
    s=2,
    color=COLOR_U
)
plt.ylabel("Tension [V]")
plt.xlabel("Date")
plt.grid()
plt.show()

plt.figure(figsize=(14,4))
plt.scatter(
    view["date"],
    view["T_C"],
    s=2,
    color=COLOR_T
)
plt.ylabel("Température [°C]")
plt.xlabel("Date")
plt.grid()
plt.show()


In [ ]:
# Zoom entre deux temps

DATE_BEGIN = "2024-06-12 20:20:00"
DATE_END = "2024-06-15 23:40:00"



t0=pd.Timestamp(
  DATE_BEGIN
)

t1=pd.Timestamp(
    DATE_END
)

zoom=df[
    (df["date"]>=t0)
    &(df["date"]<=t1)
]

plt.figure(figsize=(14,4))
plt.scatter(
    zoom["date"],
    zoom["I_A"],
    s=4,
    color=COLOR_I
)
plt.ylabel("Courant [A]")
plt.grid()
plt.show()

plt.figure(figsize=(14,4))
plt.scatter(
    zoom["date"],
    zoom["U_V"],
    s=4,
    color=COLOR_U
)
plt.ylabel("Tension [V]")
plt.grid()
plt.show()

plt.figure(figsize=(14,4))
plt.scatter(
    zoom["date"],
    zoom["T_C"],
    s=4,
    color=COLOR_T
)
plt.ylabel("Température [°C]")
plt.grid()
plt.show()



### Questions

- Où voyez-vous des phases de roulage ?
- Où voyez-vous des recharges ?
- Où voyez-vous des périodes de repos ?
- La température évolue-t-elle au cours de la saison ?
- Pourquoi les scatter plots sont-ils plus adaptés ici ?


## Les trous temporels

In [ ]:
gaps=df[
    df["delta_s"]>1
]

print(
    "Nombre de trous :",
    len(gaps)
)

gaps[[
    "date",
    "delta_s"
]].head(10)


In [ ]:
plt.figure(figsize=(14,4))
plt.scatter(
    gaps["date"],
    gaps["delta_s"]/3600,
    s=10
)
plt.ylabel("Durée du trou [h]")
plt.xlabel("Date")
plt.grid()
plt.show()


# 3. Calculs de quantités d'interêts

## Calcul de la charge et de l'énergie transférées pendant une recharge

On sélectionne une phase de recharge dans la série temporelle.

Avec la convention utilisée ici :

$$
I<0
\qquad \text{en charge}.
$$

La charge électrique transférée pendant la recharge vaut :

$$
\Delta Q
=
-\int I(t)\,dt
$$

et l'énergie transférée au pack :

$$
\Delta E
=
-\int U(t)I(t)\,dt.
$$

En pratique, avec des données discrètes :

$$
\Delta Q
\approx
-\sum_k I_k\Delta t_k
$$

et

$$
\Delta E
\approx
-\sum_k U_k I_k\Delta t_k.
$$

Si une recharge couvre tout l'intervalle

$$
SOC=0\rightarrow SOC=1,
$$

alors la charge transférée correspond directement à la capacité du pack :

$$
\boxed{C_{\max}\approx \Delta Q}
$$

Pour une recharge partielle, si le SOC initial et final sont connus :

$$
\boxed{
C_{\max}
\approx
\frac{\Delta Q}{SOC_{\mathrm{fin}}-SOC_{\mathrm{début}}}
}
$$

Remarque : dans des données d’usage réel, l’estimation de la capacité maximale à partir d’une seule recharge est difficile, car les utilisateurs exploitent rarement toute la plage de SOC. Les recharges commencent souvent avec un SOC encore relativement élevé et s’arrêtent avant 100%, ce qui ne donne accès qu’à une fraction de la capacité totale.

In [ ]:
# Zoom sur une charge

DATE_BEGIN = "2024-06-12 17:30:00"
DATE_END = "2024-06-12 21:30:00"

t0=pd.Timestamp(
  DATE_BEGIN
)

t1=pd.Timestamp(
    DATE_END
)

zoom=df[
    (df["date"]>=t0)
    &(df["date"]<=t1)
]

In [ ]:
plt.figure(figsize=(14,4))
plt.scatter(zoom["date"],zoom["I_A"],s=4,color=COLOR_I)
plt.ylabel("Courant [A]")
plt.grid()
plt.show()

plt.figure(figsize=(14,4))
plt.scatter(zoom["date"],zoom["U_V"],s=4,color=COLOR_U)
plt.ylabel("Tension [V]")
plt.grid()
plt.show()

Calculer la charge et l'énergie injectées dans cette charge. Est-ce que ça correspond à la capacité maximale ?

In [ ]:
dt_zoom=zoom["delta_s"].to_numpy()
I_zoom=zoom["I_A"].to_numpy()
U_zoom=zoom["U_V"].to_numpy()

Q_INJECTED=...
E_INJECTED=...
CAPACITY_ESTIMATED=...

print("Charge injectée :",Q_INJECTED,"Ah")
print("Énergie injectée :",E_INJECTED,"kWh")
print("Capacité estimée :",CAPACITY_ESTIMATED,"Ah")

## Bruit et relaxation


Un premier modèle très simple serait :

$$
U=OCV(SOC)-R_0I.
$$

Lorsque le courant revient à zéro, ce modèle prédit immédiatement la tension au repos :

$$
U=OCV(SOC).
$$

Les données montrent pourtant que la tension peut continuer à évoluer pendant plusieurs minutes.

Cette dynamique lente est la **relaxation**.


In [ ]:
# Zoom sur une relaxation

t0=pd.Timestamp(
    "2024-06-12 20:20:00"
)

t1=pd.Timestamp(
    "2024-06-12 23:40:00"
)

zoom=df[
    (df["date"]>=t0)
    &(df["date"]<=t1)
]

plt.figure(figsize=(14,4))
plt.scatter(
    zoom["date"],
    zoom["I_A"],
    s=4,
    color=COLOR_I
)
plt.ylabel("Courant [A]")
plt.grid()
plt.show()

plt.figure(figsize=(14,4))
plt.scatter(
    zoom["date"],
    zoom["U_V"],
    s=4,
    color=COLOR_U
)
plt.ylabel("Tension [V]")
plt.grid()
plt.show()


# 4. Modèles

## Modèle de SOC

Le SOC représente la quantité de charge disponible normalisée par la capacité maximale.

Sous l'hypothèse d'une efficacité coulombique égale à 1 :

$$
\boxed{
\frac{dSOC}{dt}
=
-\frac{I}{3600C_{\max}}
}
$$


En temps discret, avant saturation :

$$
SOC_{k+1}^{*}
=
SOC_k
-
\frac{
I_k\Delta t_k
}{
3600C_{\max}
}
$$

Le SOC est une grandeur physique bornée :

$$
0\le SOC\le1
$$

On impose donc :

$$
\boxed{
SOC_{k+1}
=
\operatorname{clip}
\left(
SOC_k
-
\frac{
I_k\Delta t_k
}{
3600C_{\max}
},
0,
1
\right)
}
$$

Il faut également connaître l'état initial :

$$
\boxed{
SOC(t=0)=SOC_0
}
$$

La trajectoire SOC dépend donc de **deux paramètres inconnus** :

$$
SOC_0
\qquad\text{et}\qquad
C_{\max}
$$

### Remarque sur l'autodiff

La fonction `clip` est différentiable tant que le SOC reste strictement dans l'intervalle :

$$
0<SOC<1
$$

Lorsque le SOC est saturé à 0 ou 1, le gradient dans la direction qui voudrait dépasser la borne devient nul.

C'est ici cohérent avec la contrainte physique du modèle.


## Courbe OCV

OCV pour open circuit voltage correspond à la tension au repos de la batterie.
On suppose ici connue la relation :

$$
OCV=OCV(SOC).
$$

Dans les données synthétiques, elle est représentée par un polynôme d'ordre 5 :

$$
OCV(SOC)
=
a_5SOC^5+a_4SOC^4+a_3SOC^3+a_2SOC^2+a_1SOC+a_0
$$

Cette fonction crée le couplage essentiel :

$$
C_{\max},SOC_0
\rightarrow
SOC(t)
\rightarrow
OCV(SOC(t))
\rightarrow
U(t)
$$

Ainsi, $C_{\max}$ et $SOC_0$ peuvent être identifiés **à partir de la tension**, sans ajouter de loss spécifique sur le SOC.


### Question
Selon la chimie, l'OCV peut ne pas être bijective (Chimie LFP par exemple). Pourquoi ça peut poser problème ?

In [ ]:
def OCV(soc):
    return jnp.polyval(
        OCV_COEF,
        soc
    )

soc_grid=jnp.linspace(
    0,
    1,
    300
)

plt.figure(figsize=(7,5))
plt.scatter(
    np.array(soc_grid),
    np.array(OCV(soc_grid)),
    s=8
)
plt.xlabel("SOC")
plt.ylabel("OCV [V]")
plt.grid()
plt.show()



## Loi vraie de $R_0(SOC,T)$ — référence synthétique

Les données actuelles ont été générer par une loi de résistance connu. On la définit maintenant afin de pouvoir comparer ensuite les fonctions identifiées.

$$
R_0(SOC,T)
=
R_{0,\min}
+
A_{SOC}(SOC-0.5)^2
+
A_T
\frac{
\operatorname{softplus}\left(\frac{25-T}{5}\right)
}{
\operatorname{softplus}\left(\frac{25-14}{5}\right)
}
$$

Elle sert uniquement de vérité terrain pour l'analyse finale.


In [ ]:
R0_MIN_TRUE=0.8
A_SOC_TRUE=0.08
A_TEMP_TRUE=0.15

def R0_true_function(soc,temp):
    num=jax.nn.softplus((25-temp)/5)
    den=jax.nn.softplus((25-14)/5)
    return R0_MIN_TRUE+A_SOC_TRUE*(soc-0.5)**2+A_TEMP_TRUE*num/den

SOC_SURF=jnp.linspace(0.1,1.0,45)
T_SURF=jnp.linspace(10,40,45)
SOC_MESH_TRUE,T_MESH_TRUE=jnp.meshgrid(SOC_SURF,T_SURF)
R0_TRUE_SURF=R0_true_function(SOC_MESH_TRUE,T_MESH_TRUE)


In [ ]:
fig=plt.figure(figsize=(9,6))
ax=fig.add_subplot(111,projection="3d")

surf=ax.plot_surface(
    np.array(SOC_MESH_TRUE),
    np.array(T_MESH_TRUE),
    np.array(R0_TRUE_SURF),
    cmap=R0_CMAP,
    linewidth=0
)

ax.set_xlabel("SOC")
ax.set_ylabel("Température [°C]")
ax.set_zlabel("R0 [Ohm]")
ax.set_title("R0 vrai")

cbar=fig.colorbar(surf,ax=ax,pad=0.12,shrink=0.75)
cbar.set_label("R0 [Ohm]")

plt.show()


### Question

Les propriétés physiques de la résistance ?

## Modèle électrique équivalent


La tension terminale est modélisée par :

$$
\boxed{
U
=
OCV(SOC)
-
R_0(SOC,T)I
-
\sum_{i=1}^{4}U_i
}
$$

Les quatre états RC suivent :

$$
\boxed{
\frac{dU_i}{dt}
=
-\frac{U_i}{\tau_i}
+
\frac{R_i}{\tau_i}I
}
$$

avec :

$$
\tau_i=R_iC_i
$$

Pour réduire le nombre de paramètres indépendants, on impose :

$$
\boxed{
\tau_i=\tau_0\,10^i
}
$$

pour $i=0,1,2,3$ et on suppose l'état initiale $U_i^0 = 0$ pour tout i.

### Question

$U_i^0 = 0$ correspond à quoi physiquement ?

## Discrétisation des états RC
On discrétise les circuits RC avec la formule approchée :
$$
U_i^{k+1}
=
e^{-\Delta t/\tau_i}U_i^k
+
\left(1-e^{-\Delta t/\tau_i}\right)R_iI_k
$$

# 5. Identification

## Identification des paramètres du modèle

Certaines grandeurs du modèle sont difficiles, voire impossibles, à mesurer directement sur un pack déjà assemblé.

À l'échelle cellule, certains paramètres électrochimiques ou électriques peuvent être estimés expérimentalement dans des conditions contrôlées. À l'échelle pack, la situation est plus complexe : la tension et le courant sont mesurables, mais des grandeurs comme la capacité effective, les constantes de temps ou les résistances internes ne sont pas directement accessibles.

La résistance observée au niveau du pack ne correspond d'ailleurs pas uniquement à celle des cellules. Elle regroupe également les contributions des connexions électriques, busbars, contacts, câblages et autres éléments du pack.

L'objectif est donc d'identifier ces paramètres à partir des séries temporelles disponibles, en comparant la tension mesurée à la tension prédite par le modèle.

On considère ici une fenêtre de données d'environ **deux semaines**. Les paramètres sont optimisés soit sous forme de **scalaires**, soit, lorsqu'une dépendance plus complexe est nécessaire, sous forme de **fonction paramétrée par un réseau de neurones**.

### Paramètres à identifier

$$
\boxed{
\theta=
\left[
SOC_0,
C_{\max},
\tau_0,
R_1,R_2,R_3,R_4,
\theta_{R_0}
\right]
}
$$

Les paramètres

$$
SOC_0,\quad
C_{\max},\quad
\tau_0,\quad
R_1,\ldots,R_4
$$

sont représentés par des scalaires.

La résistance ohmique est représentée par une fonction :

$$
R_0=R_0(SOC,T)
$$

paramétrée ici par un petit réseau de neurones.

### Hypothèses

- l'OCV est supposée connue
- le SOC initial $SOC_0$ est inconnu
- la capacité $C_{\max}$ est inconnue
- les quatre états RC sont initialisés à zéro
- les paramètres physiques restent positifs :

$$
R_i>0
\qquad
\tau_0>0
\qquad
C_{\max}>0
$$

- le SOC initial est borné :

$$
0<SOC_0<1
$$

- la résistance ohmique est bornée :

$$
0<R_0(SOC,T)<2\ \Omega
$$

## Fenêtre d'identification

Pour cette première identification, on considère une fenêtre de données d'environ **deux semaines**.

L'objectif est de disposer de suffisamment de sollicitations différentes pour identifier correctement les paramètres électriques, tout en restant sur une durée suffisamment courte pour pouvoir considérer que l'état de santé de la batterie varie peu.

Sur des périodes de plusieurs mois, le vieillissement peut en effet modifier progressivement certains paramètres du modèle, notamment $C_max$ et $R_0$

On ferait alors l'hypothèse, potentiellement fausse, que des paramètres constants décrivent toute la période.

On cherche donc ici un compromis entre fenêtre assez longue pour identifier et assez courte pour négliger le vieillissement.


In [ ]:
date_start=pd.Timestamp(
    "2024-06-16 11:19:09"
)

date_end=(
    date_start
    +pd.Timedelta(days=14)
)

train=df[
    (df["date"]>=date_start)
    &(df["date"]<date_end)
].copy()

print(
    "Points bruts :",
    len(train)
)


In [ ]:
# @title
# Réglage utilisé dans cette section
STEP=DOWNSAMPLE_STEP

train=train.iloc[::STEP].copy()

In [ ]:
I_id=jnp.array(train["I_A"])
T_id=jnp.array(train["T_C"])
U_id=jnp.array(train["U_V"])
time_id=jnp.array(train["time_s"])



dt_real=jnp.concatenate([
    jnp.array([1.0]),
    time_id[1:]-time_id[:-1]
])

GAP_MAX=MAX_GAP_S
is_gap_id=dt_real>GAP_MAX

dt_soc=jnp.where(
    is_gap_id,
    0.0,
    dt_real
)

dt_rc=dt_real

print(
    "Points utilisés :",
    len(I_id)
)

print(
    "Gaps dans la fenêtre :",
    int(jnp.sum(is_gap_id))
)


In [ ]:
SOC0_TRUE=float(
    SOC_true[
        train.index[0]
    ]
)

print("SOC0 vrai de la fenêtre :",SOC0_TRUE)

In [ ]:
plt.figure(figsize=(14,4))
plt.scatter(
    train["date"],
    train["I_A"],
    s=4,
    color=COLOR_I
)
plt.ylabel("Courant [A]")
plt.title("Fenêtre utilisée pour l'identification")
plt.grid()
plt.show()

plt.figure(figsize=(14,4))
plt.scatter(
    train["date"],
    train["U_V"],
    s=4,
    color=COLOR_U
)
plt.ylabel("Tension [V]")
plt.grid()
plt.show()

plt.figure(figsize=(14,4))
plt.scatter(
    train["date"],
    train["T_C"],
    s=4,
    color=COLOR_T
)
plt.ylabel("Température [°C]")
plt.grid()
plt.show()

## Modélisation $R_0(SOC,T)$ sous forme de réseau de neurones

On représente la résistance ohmique par un petit réseau de neurones :

$$
y_\theta
=
NN_\theta(SOC,T)
$$

La sortie brute du réseau n'a pas directement de signification physique

On applique donc une sigmoïde bornée :

$$
\boxed{
R_0(SOC,T)
=
R_{0,\max}\,
\sigma(y_\theta)
}
$$


Le réseau reste volontairement petit : le but n'est pas de construire un modèle profond, mais d'apprendre une fonction effective dépendant du SOC et de la température.


### Question
Quel est l'interêt de la sigmoide ?

Note sur Equinox :
Equinox est une bibliothèque de réseaux de neurones construite au-dessus de JAX. Elle permet de définir des modèles sous forme de classes Python tout en conservant la compatibilité avec l’autodifférentiation, la compilation JIT et les transformations JAX.

In [ ]:
# Borne supérieure du réseau R0
R0_MAX=2.0
class R0NN(eqx.Module):
    mlp:eqx.nn.MLP

    def __init__(self,key):
        self.mlp=eqx.nn.MLP(
            in_size=2,
            out_size=1,
            width_size=8,
            depth=1,
            activation=jax.nn.tanh,
            key=key
        )

    def __call__(self,soc,temp):
        x=jnp.array([soc,(temp-25)/10])
        y=self.mlp(x)[0]
        return R0_MAX*jax.nn.sigmoid(y)


In [ ]:
key=jax.random.PRNGKey(0)
r0_nn=R0NN(key)

N_PARAMS=sum(
    x.size
    for x in jax.tree_util.tree_leaves(
        eqx.filter(r0_nn,eqx.is_array)
    )
)

print("Nombre de paramètres du réseau R0 à optimiser :",N_PARAMS)

Le réseau représentant $R_0(SOC,T)$ contient seulement 33 paramètres entraînables. Il reste donc volontairement très petit par rapport au nombre de points disponibles pour l'identification.

## Modèle à identifier

In [ ]:
class ECM(eqx.Module):
    r0:R0NN
    raw_soc0:jnp.ndarray
    log_Cmax:jnp.ndarray
    log_tau0:jnp.ndarray
    log_R:jnp.ndarray

    def __init__(self,key):
        self.r0=R0NN(key)

        soc0_guess=0.65

        self.raw_soc0=jnp.array(
            jnp.log(
                soc0_guess
                /(1-soc0_guess)
            )
        )

        self.log_Cmax=jnp.array(
            jnp.log(200.0)
        )

        self.log_tau0=jnp.array(
            jnp.log(10.0)
        )

        self.log_R=jnp.log(
            jnp.array([
                0.01,
                0.01,
                0.01,
                0.01
            ])
        )



##Transformations des paramètres

Pour imposer automatiquement les contraintes :

$$
C_{\max}=e^{c_{raw}}
\qquad
\tau_0=e^{\tau_{raw}}
\qquad
R_i=e^{r_i}
$$

Toutes ces grandeurs sont donc positives pendant l'optimisation


## Visualiser le $R_0$ initial

In [ ]:
def make_r0_grid(model):
    soc_axis=jnp.linspace(
        0.1,
        1.0,
        45
    )

    temp_axis=jnp.linspace(
        10,
        40,
        45
    )

    SOC_mesh,T_mesh=jnp.meshgrid(
        soc_axis,
        temp_axis
    )

    points=jnp.stack([
        SOC_mesh.ravel(),
        T_mesh.ravel()
    ],axis=1)

    values=jax.vmap(
        lambda x:model.r0(
            x[0],
            x[1]
        )
    )(points)

    R_mesh=values.reshape(
        SOC_mesh.shape
    )

    return (
        SOC_mesh,
        T_mesh,
        R_mesh
    )


def plot_r0_surface(
    model,
    title
):
    SOC_mesh,T_mesh,R_mesh=make_r0_grid(model)

    norm=plt.Normalize(
        float(T_mesh.min()),
        float(T_mesh.max())
    )

    cmap=plt.cm.get_cmap(
        R0_CMAP
    )

    facecolors=cmap(
        norm(
            np.array(T_mesh)
        )
    )

    fig=plt.figure(
        figsize=(9,6)
    )

    ax=fig.add_subplot(
        111,
        projection="3d"
    )

    ax.plot_surface(
        np.array(SOC_mesh),
        np.array(T_mesh),
        np.array(R_mesh),
        facecolors=facecolors,
        linewidth=0,
        antialiased=True,
        shade=False
    )

    ax.set_xlabel("SOC")
    ax.set_ylabel("Température [°C]")
    ax.set_zlabel("R0 [Ohm]")
    ax.set_title(title)

    sm=plt.cm.ScalarMappable(
        norm=norm,
        cmap=cmap
    )

    sm.set_array([])

    cbar=fig.colorbar(
        sm,
        ax=ax,
        pad=0.12,
        shrink=0.75
    )

    cbar.set_label(
        "Température [°C]"
    )

    plt.show()


In [ ]:
# SEED pour changer l'aléatoiree
SEED_EXEMPLE = 5
model=ECM(
    jax.random.PRNGKey(SEED_EXEMPLE)
)

plot_r0_surface(
    model,
    "R0 initial"
);

Remarque. À ce stade, le réseau n'a encore jamais été entraîné. La surface $R_0(SOC,T)$ affichée provient uniquement de l'initialisation aléatoire de ses poids. Elle n'a donc pas encore de signification physique particulière.
En changeant la seed passée à jax.random.PRNGKey(...), on obtient une autre surface initiale.

## Simulation avec une vraie boucle Python


On commence par une implémentation directe des équations avec une boucle `for` Python.

L'état dynamique est :

$$
x_k=
\begin{bmatrix}
SOC_k\\
U_{1,k}\\
U_{2,k}\\
U_{3,k}\\
U_{4,k}
\end{bmatrix}.
$$

À chaque pas :

1. on calcule la tension au temps $k$ ;
2. on met à jour le SOC ;
3. on met à jour les quatre états RC.

Cette version est volontairement simple et lisible, mais peu efficace pour de longues séries.

In [ ]:
def simulate_python(
    model,
    I,
    T,
    dt_soc,
    dt_rc,
    is_gap
):
    soc=jax.nn.sigmoid(
        model.raw_soc0
    )

    Cmax=jnp.exp(model.log_Cmax)

    tau0=jnp.exp(model.log_tau0)

    tau=tau0*jnp.array([
        1.0,
        10.0,
        100.0,
        1000.0
    ])

    R=jnp.exp(model.log_R)

    Urc=jnp.zeros(4)

    U_pred=[]
    SOC_pred=[]

    for k in range(len(I)):
        r0=model.r0(soc,T[k])

        voltage=(
            OCV(soc)
            -r0*I[k]
            -jnp.sum(Urc)
        )

        U_pred.append(
            voltage
        )

        SOC_pred.append(
            soc
        )

        soc=(
            soc
            -I[k]
            *dt_soc[k]
            /(3600*Cmax)
        )

        soc=jnp.clip(soc,0.0,1.0)

        current_rc=jnp.where(
            is_gap[k],
            0.0,
            I[k]
        )

        alpha=jnp.exp(
            -dt_rc[k]/tau
        )

        Urc=(
            alpha*Urc
            +(1-alpha)
            *R
            *current_rc
        )

    return (
        jnp.stack(U_pred),
        jnp.stack(SOC_pred)
    )


## Même simulation avec `jax.lax.scan`

In [ ]:
def simulate_scan(
    model,
    I,
    T,
    dt_soc,
    dt_rc,
    is_gap
):
    soc0=jax.nn.sigmoid(model.raw_soc0)

    Cmax=jnp.exp(model.log_Cmax)

    tau0=jnp.exp(model.log_tau0)

    tau=tau0*jnp.array([
        1.0,
        10.0,
        100.0,
        1000.0
    ])

    R=jnp.exp(model.log_R)

    def step(
        state,
        x
    ):
        soc,Urc=state

        (
            current,
            temp,
            delta_soc,
            delta_rc,
            gap
        )=x

        r0=model.r0(soc,temp)

        voltage=(
            OCV(soc)
            -r0*current
            -jnp.sum(Urc)
        )

        soc_new=(
            soc
            -current
            *delta_soc
            /(3600*Cmax)
        )

        soc_new=jnp.clip(soc_new,0.0,1.0)

        current_rc=jnp.where(
            gap,
            0.0,
            current
        )

        alpha=jnp.exp(
            -delta_rc/tau
        )

        Urc_new=(
            alpha*Urc
            +(1-alpha)
            *R
            *current_rc
        )

        new_state=(
            soc_new,
            Urc_new
        )

        output=(
            voltage,
            soc
        )

        return (
            new_state,
            output
        )

    state0=(
        soc0,
        jnp.zeros(4)
    )

    _,outputs=jax.lax.scan(
        step,
        state0,
        (
            I,
            T,
            dt_soc,
            dt_rc,
            is_gap
        )
    )

    U_pred,SOC_pred=outputs

    return (
        U_pred,
        SOC_pred
    )



`lax.scan` représente la boucle temporelle comme une primitive JAX.

Cela permet notamment :

- l'autodiff à travers toute la trajectoire
- la compilation XLA
- une exécution beaucoup plus efficace qu'une longue boucle Python


## Loss unique sur la tension

### Fonction de coût

L'identification consiste à ajuster les paramètres du modèle afin que la tension simulée soit la plus proche possible de la tension mesurée.

Pour mesurer l'écart entre les deux, on introduit une **fonction de coût** (ou *loss*) :

$$
\mathcal L(\theta)
$$

Cette quantité dépend des paramètres à identifier :

$$
\theta=
\left[
SOC_0,
C_{\max},
\tau_0,
R_1,\ldots,R_4,
\theta_{R_0}
\right]
$$

Plus la loss est faible, plus le modèle reproduit correctement la tension mesurée.

On choisit ici l'erreur quadratique moyenne (MSE) :

$$
\boxed{
\mathcal L(\theta)
=
\frac1N
\sum_{k=1}^N
\left(
U_k^{pred}(\theta)
-
U_k^{mes}
\right)^2
}
$$

L'optimisation consiste donc à chercher :

$$
\boxed{
\theta^\star
=
\arg\min_\theta
\mathcal L(\theta)
}
$$

La loss porte uniquement sur la tension, mais cela ne signifie pas que seuls les paramètres électriques sont appris.

En effet, la tension prédite dépend du SOC simulé :

$$
U^{pred}
=
OCV(SOC^{pred})
-
R_0(SOC^{pred},T)I
-
\sum_i U_i
$$

On a donc la chaîne de dépendance :

$$
\boxed{
SOC_0,C_{\max}
\rightarrow
SOC(t)
\rightarrow
OCV(SOC)
\rightarrow
U(t)
\rightarrow
\mathcal L
}
$$

Ainsi, même sans utiliser directement une erreur sur le SOC, la loss en tension contient de l'information sur $SOC_0$ et $C_{\max}$.

In [ ]:
def loss(model):
    U_pred,_=simulate_scan(
        model,
        I_id,
        T_id,
        dt_soc,
        dt_rc,
        is_gap_id
    )

    return jnp.mean(
        (U_pred-U_id)**2
    )


In [ ]:
def r2_score(
    y_true,
    y_pred
):
    ss_res=jnp.sum(
        (y_true-y_pred)**2
    )

    ss_tot=jnp.sum(
        (
            y_true-jnp.mean(y_true)
        )**2
    )

    return (
        1
        -ss_res/ss_tot
    )


## Boucle Python, `scan` et compilation JIT

Le modèle dynamique doit être propagé sur toute la série temporelle, point après point.

Une première approche consiste à utiliser une **boucle Python classique**. Elle est simple à lire, mais peut devenir coûteuse lorsque la série contient beaucoup de points et que la simulation doit être répétée de nombreuses fois pendant l'optimisation.

On compare donc trois approches :

- une boucle Python classique ;
- `jax.lax.scan`, qui permet de représenter efficacement une dynamique séquentielle dans JAX ;
- la même simulation compilée avec JIT.

L'objectif est de comparer leur temps de calcul pour une même propagation du modèle.

Cette différence devient importante lors de l'identification : une optimisation peut nécessiter plusieurs centaines ou milliers d'itérations, et donc autant de simulations du modèle.



### Boucle classique Python

On utilise le même nombre de points pour toutes les mesures de temps.

Le réglage global est rappelé ici :

```python
N_TEST=N_TEST_SPEED
```


In [ ]:
N_TEST_SPEED = 5000
N_TEST=N_TEST_SPEED

t0=time.perf_counter()

U_python,SOC_python=simulate_python(
    model,
    I_id[:N_TEST],
    T_id[:N_TEST],
    dt_soc[:N_TEST],
    dt_rc[:N_TEST],
    is_gap_id[:N_TEST]
)

jax.block_until_ready(U_python)
TIME_PYTHON=time.perf_counter()-t0

print("Nombre de points :",N_TEST)
print("Temps boucle Python :",TIME_PYTHON,"s")



### Boucle avec `lax.scan` compilé avec JAX

Le modèle Equinox contient des tableaux entraînables mais aussi des objets statiques comme la fonction d'activation.



In [ ]:
simulate_jit=eqx.filter_jit(simulate_scan)

t0=time.perf_counter()

U_jit,SOC_jit=simulate_jit(
    model,
    I_id[:N_TEST],
    T_id[:N_TEST],
    dt_soc[:N_TEST],
    dt_rc[:N_TEST],
    is_gap_id[:N_TEST]
)

jax.block_until_ready(U_jit)
TIME_JIT_FIRST=time.perf_counter()-t0

print("Premier appel JIT :",TIME_JIT_FIRST,"s")


In [ ]:
t0=time.perf_counter()

U_jit,SOC_jit=simulate_jit(
    model,
    I_id[:N_TEST],
    T_id[:N_TEST],
    dt_soc[:N_TEST],
    dt_rc[:N_TEST],
    is_gap_id[:N_TEST]
)

jax.block_until_ready(U_jit)
TIME_JAX=time.perf_counter()-t0

SPEEDUP=TIME_PYTHON/TIME_JAX

print("Deuxième appel JAX compilé :",TIME_JAX,"s")
print("Rapport temps Python / temps JAX :",SPEEDUP)


### Question — coût d'une optimisation

Le rapport mesuré est :

$$
\text{accélération}=\frac{t_{Python}}{t_{JAX}}.
$$

Le temps mesuré correspond à une propagation du modèle. Une optimisation en demande des centaines ou milliers.  Or, pendant l’optimisation, cette simulation est répétée à chaque itération.
En supposant 1000 itérations, estimez le temps total avec la boucle Python puis avec JAX compilé.
Que concluez-vous sur l’intérêt de la compilation pour un problème d’identification ?


### À retenir

Une fonction utilisant `lax.scan` n'est pas la même chose qu'une boucle Python.

Mais `scan` est déjà une primitive JAX conçue pour être tracée et compilée.



# 6. Entraînement

In [ ]:
LEARNING_RATE = 1e-2
LR=LEARNING_RATE
optimizer=optax.adam(LR)

opt_state=optimizer.init(
    eqx.filter(
        model,
        eqx.is_array
    )
)

@eqx.filter_jit
def train_step(
    model,
    opt_state
):
    value,grads=(
        eqx.filter_value_and_grad(
            loss
        )(model)
    )

    updates,opt_state=(
        optimizer.update(
            grads,
            opt_state,
            model
        )
    )

    model=eqx.apply_updates(
        model,
        updates
    )

    return (
        model,
        opt_state,
        value
    )


### Évolution de $R_0(SOC,T)$ pendant l'optimisation

In [ ]:
from IPython.display import clear_output

history=[]
r2_history=[]
# Réglages utilisés pour cet entraînement
N_EPOCHS=300
DISPLAY_EVERY=DISPLAY_R0_EVERY
DISPLAY_UNTIL=DISPLAY_R0_UNTIL
PRINT_EVERY=10
PRINT_STEP=PRINT_EVERY
DISPLAY_R0_EVERY=2
DISPLAY_R0_UNTIL=30


for epoch in range(
    N_EPOCHS
):
    model,opt_state,value=train_step(
        model,
        opt_state
    )

    history.append(
        float(value)
    )

    if (
        epoch%DISPLAY_EVERY==0
        and epoch<=DISPLAY_UNTIL
    ):
        U_epoch,_=simulate_scan(
            model,
            I_id,
            T_id,
            dt_soc,
            dt_rc,
            is_gap_id
        )

        r2_epoch=float(
            r2_score(
                U_id,
                U_epoch
            )
        )

        r2_history.append(
            [
                epoch,
                r2_epoch
            ]
        )

        clear_output(
            wait=True
        )

        soc0_now=float(
            jax.nn.sigmoid(
                model.raw_soc0
            )
        )

        cmax_now=float(
            jnp.exp(
                model.log_Cmax
            )
        )

        print(
            "epoch",
            epoch,
            "/",
            N_EPOCHS
        )

        print(
            "loss =",
            float(value)
        )

        print(
            "R² =",
            r2_epoch
        )

        print(
            "SOC0 =",
            soc0_now
        )

        print(
            "Cmax =",
            cmax_now,
            "Ah"
        )

        plot_r0_surface(
            model,
            "R0 après epoch "
            +str(epoch)
        )

    elif epoch%PRINT_STEP==0:
        U_epoch,_=simulate_scan(
            model,
            I_id,
            T_id,
            dt_soc,
            dt_rc,
            is_gap_id
        )

        r2_epoch=float(
            r2_score(
                U_id,
                U_epoch
            )
        )

        r2_history.append(
            [
                epoch,
                r2_epoch
            ]
        )

        print(
            "epoch",
            epoch,
            "loss",
            float(value),
            "R²",
            r2_epoch
        )


In [ ]:
plt.figure(figsize=(7,4))
plt.scatter(
    np.arange(len(history)),
    history,
    s=8,
    color=COLOR_U
)
plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("MSE tension")
plt.grid()
plt.show()


Discussion sur la valeur de la loss ?


### Suivi du coefficient $R^2$

En complément de la MSE, on suit :

$$
R^2
=
1-
\frac{
\sum_k
\left(
U_k^{mes}-U_k^{pred}
\right)^2
}{
\sum_k
\left(
U_k^{mes}-\overline U^{mes}
\right)^2
}.
$$

Un $R^2$ proche de 1 signifie que le modèle reproduit une grande partie de la variabilité de la tension mesurée.


In [ ]:
r2_history=np.array(
    r2_history
)

plt.figure(figsize=(7,4))

plt.scatter(
    r2_history[:,0],
    r2_history[:,1],
    s=20,
    color=COLOR_U
)

plt.xlabel("Epoch")
plt.ylabel("R²")
plt.grid()
plt.show()


## Comparaison visuelle de la tension

In [ ]:
U_pred,SOC_pred=simulate_scan(
    model,
    I_id,
    T_id,
    dt_soc,
    dt_rc,
    is_gap_id
)

R2_voltage=float(
    r2_score(
        U_id,
        U_pred
    )
)

print(
    "R² tension global :",
    R2_voltage
)

n=15000

plt.figure(figsize=(14,4))
plt.scatter(
    np.arange(n),
    np.array(U_id[:n]),
    s=3,
    color=COLOR_U,
    label="Mesure"
)
plt.scatter(
    np.arange(n),
    np.array(U_pred[:n]),
    s=3,
    color=COLOR_I,
    label="Modèle"
)
plt.ylabel("Tension [V]")
plt.xlabel("Point")
plt.legend()
plt.grid()
plt.show()



### Zoom local sur la tension prédite

La vue globale permet de vérifier le comportement sur une longue durée, mais elle masque les détails.

On zoome ici sur une courte fenêtre pour vérifier si le modèle reproduit :

- les variations rapides liées au courant
- les phases de relaxation
- les petits écarts de tension.


In [ ]:
ZOOM_END=ZOOM_START+ZOOM_LENGTH
x_zoom=np.arange(ZOOM_START,ZOOM_END)



plt.figure(figsize=(14,5))

plt.scatter(
    x_zoom,
    np.array(
        U_id[ZOOM_START:ZOOM_END]
    ),
    s=8,
    color=COLOR_U,
    label="Mesure"
)

plt.scatter(
    x_zoom,
    np.array(
        U_pred[ZOOM_START:ZOOM_END]
    ),
    s=8,
    color=COLOR_I,
    label="Prédiction"
)

plt.xlabel("Point")
plt.ylabel("Tension [V]")
plt.title(
    "Zoom mesure vs prédiction"
)

plt.legend()
plt.grid()
plt.show()

R2_zoom=float(
    r2_score(
        U_id[ZOOM_START:ZOOM_END],
        U_pred[ZOOM_START:ZOOM_END]
    )
)

print(
    "R² sur le zoom :",
    R2_zoom
)


In [ ]:
error=(
    U_pred-U_id
)

plt.figure(figsize=(14,4))
plt.scatter(
    np.arange(n),
    np.array(error[:n]),
    s=3
)
plt.axhline(
    0,
    linestyle="--"
)
plt.ylabel("Erreur [V]")
plt.xlabel("Point")
plt.grid()
plt.show()



## Analyse statistique des résidus

On définit le résidu :

$$
\varepsilon_k
=
U_k^{pred}
-
U_k^{mes}.
$$

Si le modèle capture correctement la dynamique moyenne, on souhaite idéalement :

$$
\mathbb E[\varepsilon]\approx0.
$$

On regarde également l'écart-type :

$$
\sigma_\varepsilon
=
\sqrt{
\mathbb E[
(\varepsilon-\bar\varepsilon)^2
]
}.
$$

Comme le jeu synthétique contient un bruit de tension connu, on pourra comparer l'écart-type des résidus au niveau de bruit injecté.


In [ ]:
residual=np.array(
    U_pred-U_id
)

res_mean=float(
    np.mean(residual)
)

res_std=float(
    np.std(residual)
)

res_rmse=float(
    np.sqrt(
        np.mean(
            residual**2
        )
    )
)

print(
    "Moyenne résidu :",
    res_mean,
    "V"
)

print(
    "Écart-type résidu :",
    res_std,
    "V"
)

print(
    "RMSE :",
    res_rmse,
    "V"
)

print(
    "Bruit injecté :",
    VOLTAGE_NOISE_STD,
    "V"
)


R2_global=float(
    r2_score(
        U_id,
        U_pred
    )
)

print(
    "R² global :",
    R2_global
)


In [ ]:
plt.figure(figsize=(8,5))

plt.hist(
    residual,
    bins=80,
    density=True,
    alpha=0.8,
    color=COLOR_U
)

plt.axvline(
    res_mean,
    linestyle="--",
    color=COLOR_I,
    label="Moyenne"
)

plt.xlabel("Résidu U_pred - U_mes [V]")
plt.ylabel("Densité")
plt.title("Distribution des résidus")
plt.legend()
plt.grid()
plt.show()


## Paramètres identifiés

In [ ]:
SOC0_id=jax.nn.sigmoid(
    model.raw_soc0
)

Cmax_id=jnp.exp(
    model.log_Cmax
)

tau0_id=jnp.exp(
    model.log_tau0
)

tau_id=(
    tau0_id
    *jnp.array([
        1.0,
        10.0,
        100.0,
        1000.0
    ])
)

R_id=jnp.exp(
    model.log_R
)

print(
    "SOC0 identifié :",
    float(SOC0_id)
)

print(
    "Cmax identifié :",
    float(Cmax_id),
    "Ah"
)

print(
    "tau0 identifié :",
    float(tau0_id),
    "s"
)

print(
    "tau identifiés :",
    tau_id
)

print(
    "R identifiés :",
    R_id
)



## Validation de la calibration

On compare :

- les paramètres identifiés aux vraies valeurs
- le SOC prédit au SOC vrai
- la fonction $R_0(SOC,T)$ prédite à la vraie fonction


In [ ]:
PACK_SINGLE=pd.DataFrame({
    "Grandeur":[
        "SOC0","Cmax [Ah]","tau0 [s]",
        "R1 [Ohm]","R2 [Ohm]","R3 [Ohm]","R4 [Ohm]"
    ],
    "Vrai":[
        SOC0_TRUE,C_MAX_TRUE,TAU0_TRUE,
        float(R_RC_TRUE[0]),float(R_RC_TRUE[1]),
        float(R_RC_TRUE[2]),float(R_RC_TRUE[3])
    ],
    "Prédit":[
        float(SOC0_id),float(Cmax_id),float(tau0_id),
        float(R_id[0]),float(R_id[1]),
        float(R_id[2]),float(R_id[3])
    ]
})

PACK_SINGLE["Erreur relative [%]"]=100*np.abs(
    PACK_SINGLE["Prédit"]-PACK_SINGLE["Vrai"]
)/np.maximum(np.abs(PACK_SINGLE["Vrai"]),1e-12)

PACK_SINGLE


Les paramètres identifiés sont globalement du bon ordre de grandeur, mais certaines erreurs restent importantes. La capacité $C_{\max}$ est relativement bien retrouvée, avec une erreur d’environ 4%, alors que $SOC_0$, $\tau_0$ et certaines résistances RC présentent des écarts plus marqués.
Cela montre qu’une bonne reproduction de la tension ne garantit pas nécessairement une identification parfaite de tous les paramètres internes. Plusieurs combinaisons de paramètres peuvent produire des tensions proches : c’est une première indication d’un problème d’identifiabilité.
Ces résultats pourraient probablement être améliorés en augmentant le nombre d’itérations, en ajustant le learning rate, en modifiant l’initialisation ou encore en utilisant une stratégie d’optimisation plus adaptée. La longueur et la richesse de la fenêtre d’identification jouent également un rôle important.

Une très bonne prédiction de la tension ne garantit pas que les paramètres identifiés soient physiquement corrects.
Plusieurs combinaisons de paramètres peuvent parfois produire des tensions très proches :
$$
\boxed{
U^{pred}\approx U^{mes}
}
$$
tout en donnant des valeurs différentes de
$$
SOC_0,\quad C_{\max},\quad \tau_i,\quad R_i,\quad R_0.
$$
Le modèle peut donc être très bon en prédiction, mais moins fiable pour l'interprétation physique des paramètres.
C'est précisément ce qu'on appelle un problème d'identifiabilité : les données observées ne permettent pas toujours de déterminer de manière unique tous les paramètres internes du modèle.


### $R_0$ vrai vs prédit


In [ ]:
from matplotlib.patches import Patch

SOC_MESH,T_MESH,R0_SINGLE=make_r0_grid(model)
R0_REF=R0_true_function(SOC_MESH,T_MESH)

fig=plt.figure(figsize=(10,7))
ax=fig.add_subplot(111,projection="3d")

ax.plot_surface(
    np.array(SOC_MESH),np.array(T_MESH),np.array(R0_REF),
    cmap="viridis",alpha=0.55,linewidth=0
)

ax.plot_surface(
    np.array(SOC_MESH),np.array(T_MESH),np.array(R0_SINGLE),
    cmap="plasma",alpha=0.55,linewidth=0
)

ax.set_xlabel("SOC")
ax.set_ylabel("Température [°C]")
ax.set_zlabel("R0 [Ohm]")
ax.set_title("R0 vrai vs prédit")

legend_handles=[
    Patch(facecolor=plt.cm.viridis(0.6),alpha=0.55,label="Vrai"),
    Patch(facecolor=plt.cm.plasma(0.6),alpha=0.55,label="Prédit")
]
ax.legend(handles=legend_handles,loc="upper left")

plt.show()

R0_ERR_SINGLE=R0_SINGLE-R0_REF

print("MAE surface R0 :",float(jnp.mean(jnp.abs(R0_ERR_SINGLE))),"Ohm")
print("RMSE surface R0 :",float(jnp.sqrt(jnp.mean(R0_ERR_SINGLE**2))),"Ohm")


## Vérification du SOC — seulement après l'identification

On peut maintenant le comparer à la trajectoire reconstruite pour vérifier si l'identification de $SOC_0$ et $C_{\max}$ est cohérente.


In [ ]:
SOC_TRUE_SINGLE=SOC_true[train.index.to_numpy()]
SOC_ERROR_SINGLE=np.array(SOC_pred-SOC_TRUE_SINGLE)

SOC_MAE_SINGLE=float(np.mean(np.abs(SOC_ERROR_SINGLE)))
SOC_RMSE_SINGLE=float(np.sqrt(np.mean(SOC_ERROR_SINGLE**2)))

plt.figure(figsize=(14,5))
plt.scatter(np.arange(len(SOC_TRUE_SINGLE)),np.array(SOC_TRUE_SINGLE),
            s=3,color=COLOR_SOC,label="SOC vrai")
plt.scatter(np.arange(len(SOC_pred)),np.array(SOC_pred),
            s=3,color="purple",label="SOC prédit")

plt.ylim(-0.02,1.02)
plt.xlabel("Point")
plt.ylabel("SOC")
plt.title("SOC vrai vs SOC prédit")
plt.legend()
plt.grid()
plt.show()

print("MAE SOC :",SOC_MAE_SINGLE)
print("RMSE SOC :",SOC_RMSE_SINGLE)


Le SOC reconstruit suit globalement correctement le SOC vrai, mais on observe un décalage progressif sur certaines portions de la trajectoire, notamment vers la fin.
Avec un seul $SOC_0$, l'optimisation cherche en quelque sorte un état initial global qui convient au mieux à toute la fenêtre. Or, comme le SOC est obtenu par intégration du courant,
$$
SOC(t)
=
SOC_0
-
\frac{1}{3600C_{\max}}
\int I(t)\,dt,
$$
une petite erreur sur $SOC_0$, $C_{\max}$ ou sur le modèle peut s'accumuler progressivement et produire un drift du SOC.
Sur une trajectoire longue, il peut donc être pertinent de recaler périodiquement le SOC plutôt que d'utiliser un unique $SOC_0$ pour toute la fenêtre.


# 7. Entrainement en multiple shooting

Le SOC reconstruit suit correctement la tendance générale, mais on observe progressivement un **décalage**.

Ce phénomène vient du fait que le SOC est obtenu par intégration :

$$
SOC_k
=
SOC_0
-
\frac{1}{3600C_{\max}}
\sum_{j=0}^{k-1}
I_j\Delta t_j.
$$

Une petite erreur sur :

$$
SOC_0
$$

ou sur :

$$
C_{\max}
$$

peut donc s'accumuler au cours du temps.

On va maintenant corriger ce drift en introduisant un deuxième état initial de SOC au milieu de la trajectoire.



## Principe du double shooting

On coupe la série en deux fenêtres :

$$
[0,t_m]
\qquad\text{et}\qquad
[t_m,t_f]
$$

On introduit deux états initiaux :

$$
SOC_0^{(1)}
\qquad\text{et}\qquad
SOC_0^{(2)}
$$

Tous les autres paramètres restent globaux :

$$
C_{\max},
\quad
\tau_0,
\quad
R_1,\ldots,R_4,
\quad
R_0(SOC,T)
$$

La première fenêtre part de :

$$
SOC(0)=SOC_0^{(1)}
$$

et la seconde de :

$$
SOC(t_m)=SOC_0^{(2)}
$$


In [ ]:
CUT=50_000

I_1=I_id[:CUT]
T_1=T_id[:CUT]
U_1=U_id[:CUT]
dt_soc_1=dt_soc[:CUT]
dt_rc_1=dt_rc[:CUT]
gap_1=is_gap_id[:CUT]

I_2=I_id[CUT:]
T_2=T_id[CUT:]
U_2=U_id[CUT:]
dt_soc_2=dt_soc[CUT:]
dt_rc_2=dt_rc[CUT:]
gap_2=is_gap_id[CUT:]

print(
    "Points fenêtre 1 :",
    len(I_1)
)

print(
    "Points fenêtre 2 :",
    len(I_2)
)



## Modèle à deux $SOC_0$

On apprend maintenant :

$$
\theta=
\left[
SOC_0^{(1)},
SOC_0^{(2)},
C_{\max},
\tau_0,
R_1,\ldots,R_4,
\theta_{R_0}
\right]
$$


In [ ]:
class ECM2Shoot(eqx.Module):
    r0:R0NN
    raw_soc01:jnp.ndarray
    raw_soc02:jnp.ndarray
    log_Cmax:jnp.ndarray
    log_tau0:jnp.ndarray
    log_R:jnp.ndarray

    def __init__(
        self,
        key
    ):
        self.r0=R0NN(
            key
        )

        soc01_guess=0.65
        soc02_guess=0.65

        self.raw_soc01=jnp.array(
            jnp.log(
                soc01_guess
                /(1-soc01_guess)
            )
        )

        self.raw_soc02=jnp.array(
            jnp.log(
                soc02_guess
                /(1-soc02_guess)
            )
        )

        self.log_Cmax=jnp.array(
            jnp.log(
                200.0
            )
        )

        self.log_tau0=jnp.array(
            jnp.log(
                10.0
            )
        )

        self.log_R=jnp.log(
            jnp.array([
                0.01,
                0.01,
                0.01,
                0.01
            ])
        )


In [ ]:
def simulate_window(
    model,
    raw_soc0,
    I,
    T,
    dt_soc,
    dt_rc,
    is_gap
):
    soc0=jax.nn.sigmoid(raw_soc0)

    Cmax=jnp.exp(model.log_Cmax)

    tau0=jnp.exp(model.log_tau0)

    tau=tau0*jnp.array([
        1.0,
        10.0,
        100.0,
        1000.0
    ])

    R=jnp.exp(model.log_R)

    def step(
        state,
        x
    ):
        soc,Urc=state

        (
            current,
            temp,
            delta_soc,
            delta_rc,
            gap
        )=x

        r0=model.r0(soc,temp)

        voltage=(
            OCV(soc)
            -r0*current
            -jnp.sum(Urc)
        )

        soc_new=(
            soc
            -current
            *delta_soc
            /(3600*Cmax)
        )

        soc_new=jnp.clip(soc_new,0.0,1.0)

        current_rc=jnp.where(
            gap,
            0.0,
            current
        )

        alpha=jnp.exp(
            -delta_rc/tau
        )

        Urc_new=(
            alpha*Urc
            +(1-alpha)
            *R
            *current_rc
        )

        return (
            (
                soc_new,
                Urc_new
            ),
            (
                voltage,
                soc
            )
        )

    state0=(
        soc0,
        jnp.zeros(4)
    )

    final_state,outputs=jax.lax.scan(
        step,
        state0,
        (
            I,
            T,
            dt_soc,
            dt_rc,
            is_gap
        )
    )

    U_pred,SOC_pred=outputs

    soc_end=final_state[0]

    return (
        U_pred,
        SOC_pred,
        soc_end
    )



## Régularisation de continuité

Si les deux SOC initiaux sont totalement indépendants, le second peut absorber artificiellement une partie de l'erreur du modèle.

On ajoute donc :

$$
\mathcal L_{\mathrm{cont}}
=
\left(
SOC_{\mathrm{fin}}^{(1)}
-
SOC_0^{(2)}
\right)^2.
$$

La loss totale devient :

$$
\boxed{
\mathcal L
=
\mathcal L_U
+
\lambda_{\mathrm{cont}}
\mathcal L_{\mathrm{cont}}
}
$$



In [ ]:
# Réglage utilisé pour la continuité
LAMBDA=LAMBDA_CONT

LAMBDA_CONT=1e3

def loss_2shoot(
    model
):
    (
        U_pred_1,
        _,
        soc_end_1
    )=simulate_window(
        model,
        model.raw_soc01,
        I_1,
        T_1,
        dt_soc_1,
        dt_rc_1,
        gap_1
    )

    (
        U_pred_2,
        _,
        _
    )=simulate_window(
        model,
        model.raw_soc02,
        I_2,
        T_2,
        dt_soc_2,
        dt_rc_2,
        gap_2
    )

    loss_1=jnp.mean(
        (U_pred_1-U_1)**2
    )

    loss_2=jnp.mean(
        (U_pred_2-U_2)**2
    )

    loss_U=(
        0.5
        *(loss_1+loss_2)
    )

    soc02=jax.nn.sigmoid(
        model.raw_soc02
    )

    loss_cont=(
        soc_end_1
        -soc02
    )**2

    total=(
        loss_U
        +LAMBDA_CONT
        *loss_cont
    )

    return (
        total,
        loss_U,
        loss_cont
    )


In [ ]:
LR_DOUBLE=LEARNING_RATE
optimizer_ms=optax.adam(LR_DOUBLE)

def scalar_loss_2shoot(
    model
):
    return loss_2shoot(
        model
    )[0]

@eqx.filter_jit
def train_step_2shoot(
    model,
    opt_state
):
    value,grads=(
        eqx.filter_value_and_grad(
            scalar_loss_2shoot
        )(model)
    )

    updates,opt_state=optimizer_ms.update(
        grads,
        opt_state,
        model
    )

    model=eqx.apply_updates(
        model,
        updates
    )

    return (
        model,
        opt_state,
        value
    )



## Entraînement du double shooting


In [ ]:
model_2shoot=ECM2Shoot(
    jax.random.PRNGKey(10)
)

opt_state_2shoot=optimizer_ms.init(
    eqx.filter(
        model_2shoot,
        eqx.is_array
    )
)

history_2shoot=[]

# Réglage utilisé pour le double shooting
N_EPOCHS_2SHOOT=N_EPOCHS_DOUBLE

for epoch in range(
    N_EPOCHS_2SHOOT
):
    (
        model_2shoot,
        opt_state_2shoot,
        value_2shoot
    )=train_step_2shoot(
        model_2shoot,
        opt_state_2shoot
    )

    history_2shoot.append(
        float(value_2shoot)
    )

    if epoch%100==0:
        (
            total,
            loss_u,
            loss_cont
        )=loss_2shoot(
            model_2shoot
        )

        print(
            "epoch",
            epoch,
            "loss",
            float(total),
            "loss_U",
            float(loss_u),
            "loss_cont",
            float(loss_cont)
        )


## Comparaison visuelle de la tension

In [ ]:
(
    U_2s_1,
    SOC_2s_1,
    soc_end_1
)=simulate_window(
    model_2shoot,
    model_2shoot.raw_soc01,
    I_1,
    T_1,
    dt_soc_1,
    dt_rc_1,
    gap_1
)

(
    U_2s_2,
    SOC_2s_2,
    _
)=simulate_window(
    model_2shoot,
    model_2shoot.raw_soc02,
    I_2,
    T_2,
    dt_soc_2,
    dt_rc_2,
    gap_2
)

U_2shoot=jnp.concatenate([
    U_2s_1,
    U_2s_2
])

SOC_2shoot=jnp.concatenate([
    SOC_2s_1,
    SOC_2s_2
])

R2_2shoot=float(
    r2_score(
        U_id,
        U_2shoot
    )
)

RMSE_2shoot=float(
    jnp.sqrt(
        jnp.mean(
            (U_2shoot-U_id)**2
        )
    )
)

print(
    "R² double shooting :",
    R2_2shoot
)

print(
    "RMSE double shooting :",
    RMSE_2shoot,
    "V"
)


In [ ]:
n=15000

plt.figure(figsize=(14,4))

plt.scatter(
    np.arange(n),
    np.array(U_id[:n]),
    s=3,
    color=COLOR_U,
    label="Mesure"
)

plt.scatter(
    np.arange(n),
    np.array(U_2shoot[:n]),
    s=3,
    color=COLOR_I,
    label="Double shooting"
)

plt.ylabel("Tension [V]")
plt.xlabel("Point")
plt.legend()
plt.grid()
plt.show()



### Zoom local


In [ ]:
ZOOM_END=ZOOM_START+ZOOM_LENGTH
x_zoom=np.arange(ZOOM_START,ZOOM_END)



plt.figure(figsize=(14,5))

plt.scatter(
    x_zoom,
    np.array(
        U_id[ZOOM_START:ZOOM_END]
    ),
    s=8,
    color=COLOR_U,
    label="Mesure"
)

plt.scatter(
    x_zoom,
    np.array(
        U_2shoot[ZOOM_START:ZOOM_END]
    ),
    s=8,
    color=COLOR_I,
    label="Double shooting"
)

plt.xlabel("Point")
plt.ylabel("Tension [V]")
plt.legend()
plt.grid()
plt.show()

print(
    "R² zoom :",
    float(
        r2_score(
            U_id[
                ZOOM_START:ZOOM_END
            ],
            U_2shoot[ZOOM_START:ZOOM_END]
        )
    )
)



## Résidus du double shooting




In [ ]:
residual_2shoot=np.array(
    U_2shoot-U_id
)

mean_2shoot=float(
    np.mean(
        residual_2shoot
    )
)

std_2shoot=float(
    np.std(
        residual_2shoot
    )
)

rmse_2shoot=float(
    np.sqrt(
        np.mean(
            residual_2shoot**2
        )
    )
)

print(
    "Moyenne résidu :",
    mean_2shoot,
    "V"
)

print(
    "Écart-type résidu :",
    std_2shoot,
    "V"
)

print(
    "RMSE :",
    rmse_2shoot,
    "V"
)

print(
    "R² :",
    R2_2shoot
)

print(
    "Bruit injecté :",
    VOLTAGE_NOISE_STD,
    "V"
)


In [ ]:
plt.figure(figsize=(8,5))

plt.hist(
    residual_2shoot,
    bins=80,
    density=True,
    alpha=0.8,
    color=COLOR_U
)

plt.axvline(
    mean_2shoot,
    linestyle="--",
    color=COLOR_I,
    label="Moyenne"
)

plt.xlabel(
    "Résidu U_pred - U_mes [V]"
)

plt.ylabel("Densité")
plt.title(
    "Résidus du double shooting"
)

plt.legend()
plt.grid()
plt.show()



## Surface $R_0$ obtenue avec le double shooting


In [ ]:
plot_r0_surface(
    model_2shoot,
    "R0 appris — double shooting"
)



## SOC reconstruit avec le double shooting


In [ ]:
SOC_TRUE_DOUBLE=SOC_true[
    train.index.to_numpy()
]

SOC_ERROR_DOUBLE=np.array(SOC_2shoot-SOC_TRUE_DOUBLE)

SOC_MAE_DOUBLE=float(np.mean(np.abs(SOC_ERROR_DOUBLE)))
SOC_RMSE_DOUBLE=float(np.sqrt(np.mean(SOC_ERROR_DOUBLE**2)))


plt.figure(figsize=(14,5))

plt.scatter(
    np.arange(
        len(SOC_TRUE_DOUBLE)
    ),
    np.array(
        SOC_TRUE_DOUBLE
    ),
    s=3,
    color=COLOR_SOC,
    label="SOC vrai"
)

plt.scatter(
    np.arange(
        len(SOC_2shoot)
    ),
    np.array(
        SOC_2shoot
    ),
    s=3,
    color="purple",
    label="SOC double shooting"
)

plt.axvline(
    CUT,
    linestyle="--",
    color="black",
    label="Séparation"
)

plt.ylim(
    -0.02,
    1.02
)

plt.xlabel("Point")
plt.ylabel("SOC")
plt.legend()
plt.grid()
plt.show()


In [ ]:
soc01_2shoot=float(
    jax.nn.sigmoid(
        model_2shoot.raw_soc01
    )
)

soc02_2shoot=float(
    jax.nn.sigmoid(
        model_2shoot.raw_soc02
    )
)

print(
    "SOC01 identifié :",
    soc01_2shoot
)

print(
    "SOC02 identifié :",
    soc02_2shoot
)

print(
    "SOC fin fenêtre 1 :",
    float(soc_end_1)
)

print(
    "Écart de continuité :",
    float(
        soc02_2shoot
        -soc_end_1
    )
)



## Validation du double shooting

On termine maintenant complètement l'étude du **double shooting** avant de passer à l'identifiabilité.

On compare exactement les mêmes éléments que pour le single shooting :

- paramètres vrais et prédits
- SOC vrai et prédit
- $R_0(SOC,T)$ vrai et prédit


In [ ]:
Cmax_2shoot=jnp.exp(model_2shoot.log_Cmax)
tau0_2shoot=jnp.exp(model_2shoot.log_tau0)
R_2shoot_id=jnp.exp(model_2shoot.log_R)

SOC02_TRUE=float(SOC_TRUE_DOUBLE[CUT])

PACK_DOUBLE=pd.DataFrame({
    "Grandeur":[
        "SOC01","SOC02","Cmax [Ah]","tau0 [s]",
        "R1 [Ohm]","R2 [Ohm]","R3 [Ohm]","R4 [Ohm]"
    ],
    "Vrai":[
        SOC0_TRUE,SOC02_TRUE,C_MAX_TRUE,TAU0_TRUE,
        float(R_RC_TRUE[0]),float(R_RC_TRUE[1]),
        float(R_RC_TRUE[2]),float(R_RC_TRUE[3])
    ],
    "Prédit":[
        soc01_2shoot,soc02_2shoot,float(Cmax_2shoot),float(tau0_2shoot),
        float(R_2shoot_id[0]),float(R_2shoot_id[1]),
        float(R_2shoot_id[2]),float(R_2shoot_id[3])
    ]
})

PACK_DOUBLE["Erreur relative [%]"]=100*np.abs(
    PACK_DOUBLE["Prédit"]-PACK_DOUBLE["Vrai"]
)/np.maximum(np.abs(PACK_DOUBLE["Vrai"]),1e-12)

PACK_DOUBLE



### $R_0$ vrai vs prédit — double shooting


In [ ]:
SOC_MESH,T_MESH,R0_DOUBLE=make_r0_grid(model_2shoot)
R0_REF=R0_true_function(SOC_MESH,T_MESH)

fig=plt.figure(figsize=(10,7))
ax=fig.add_subplot(111,projection="3d")

ax.plot_surface(
    np.array(SOC_MESH),np.array(T_MESH),np.array(R0_REF),
    cmap="viridis",alpha=0.55,linewidth=0
)

ax.plot_surface(
    np.array(SOC_MESH),np.array(T_MESH),np.array(R0_DOUBLE),
    cmap="plasma",alpha=0.55,linewidth=0
)

ax.set_xlabel("SOC")
ax.set_ylabel("Température [°C]")
ax.set_zlabel("R0 [Ohm]")
ax.set_title("Double shooting : R0 vrai vs prédit")

legend_handles=[
    Patch(facecolor=plt.cm.viridis(0.6),alpha=0.55,label="Vrai"),
    Patch(facecolor=plt.cm.plasma(0.6),alpha=0.55,label="Prédit")
]
ax.legend(handles=legend_handles,loc="upper left")

plt.show()

R0_ERR_DOUBLE=R0_DOUBLE-R0_REF

print("MAE surface R0 :",float(jnp.mean(jnp.abs(R0_ERR_DOUBLE))),"Ohm")
print("RMSE surface R0 :",float(jnp.sqrt(jnp.mean(R0_ERR_DOUBLE**2))),"Ohm")


Le passage au double shooting améliore nettement certains paramètres. En particulier, l'erreur sur la capacité devient meilleure.

On observe aussi une meilleure reconstruction de la fonction $R_0$ par rapport au single shooting. Le recalage du SOC à mi-trajectoire réduit donc une partie du drift et limite la compensation de certaines erreurs par les paramètres électriques.
Cela montre qu'une meilleure gestion des états initiaux peut améliorer à la fois la reconstruction du SOC et l'identification des paramètres physiques.

### Question
le premier $SOC_0$ reste peu précis. Comment pourrait-on améliorer encore la reconstruction du SOC sur une longue trajectoire ?


##  Bilan single shooting / double shooting

On compare les deux formulations avec les mêmes indicateurs.


In [ ]:
SUMMARY_SHOOTING=pd.DataFrame({
    "Méthode":["Single shooting","Double shooting"],
    "R2 tension":[R2_voltage,R2_2shoot],
    "RMSE U [V]":[res_rmse,RMSE_2shoot],
    "MAE SOC":[SOC_MAE_SINGLE,SOC_MAE_DOUBLE],
    "RMSE SOC":[SOC_RMSE_SINGLE,SOC_RMSE_DOUBLE]
})

SUMMARY_SHOOTING


# 8. Identifiabilité et sensibilité à l'initialisation

Le modèle double shooting est maintenant fixé.

On répète exactement la même optimisation avec plusieurs initialisations aléatoires afin d'étudier la sensibilité du problème inverse.

On compare :

$$
SOC_0^{(1)},\quad
SOC_0^{(2)},\quad
C_{\max},\quad
\tau_0,\quad
R_i,\quad
R_0(SOC,T),\quad
R^2
$$

Une bonne qualité de prédiction de la tension ne garantit pas nécessairement que les mêmes paramètres physiques soient retrouvés.



On répète maintenant l'identification avec plusieurs initialisations.

La question n'est pas seulement :

> Est-ce que la tension est bien prédite ?

mais aussi :

> Est-ce que les mêmes paramètres physiques sont retrouvés ?

Deux ensembles de paramètres peuvent parfois donner des tensions très proches.

C'est un problème d'**identifiabilité pratique**.


In [ ]:
def train_double(seed,n_epochs=500):
    print(f"\n===== Initialisation {seed+1}/{N_IDENT_RUNS} =====")

    model=ECM2Shoot(jax.random.PRNGKey(seed))
    opt_state=optimizer_ms.init(eqx.filter(model,eqx.is_array))

    for epoch in range(n_epochs):
        model,opt_state,value=train_step_2shoot(model,opt_state)

        if epoch%50==0 or epoch==n_epochs-1:
            print(
                f"Epoch {epoch:4d} | "
                f"Loss = {float(value):.6f}"
            )

    SOC01=jax.nn.sigmoid(model.raw_soc01)
    SOC02=jax.nn.sigmoid(model.raw_soc02)
    CMAX=jnp.exp(model.log_Cmax)
    TAU0=jnp.exp(model.log_tau0)
    R=jnp.exp(model.log_R)

    return model,value,SOC01,SOC02,CMAX,TAU0,R

In [ ]:
results=[]
models_ident=[]

N_EPOCHS_IDENT = 250

for seed in range(N_IDENT_RUNS):

    model_seed,value,SOC01,SOC02,CMAX,TAU0,R=train_double(
        seed,
        n_epochs=N_EPOCHS_IDENT
    )

    U1,_,_=simulate_window(
        model_seed,model_seed.raw_soc01,
        I_1,T_1,dt_soc_1,dt_rc_1,gap_1
    )

    U2,_,_=simulate_window(
        model_seed,model_seed.raw_soc02,
        I_2,T_2,dt_soc_2,dt_rc_2,gap_2
    )

    U_seed=jnp.concatenate([U1,U2])

    R2_VOLTAGE=float(
        r2_score(U_id,U_seed)
    )

    models_ident.append(model_seed)

    results.append({
        "Run":seed+1,
        "Loss":float(value),
        "R2 tension":R2_VOLTAGE,
        "SOC01":float(SOC01),
        "SOC02":float(SOC02),
        "Cmax [Ah]":float(CMAX),
        "tau0 [s]":float(TAU0),
        "R1 [Ohm]":float(R[0]),
        "R2 [Ohm]":float(R[1]),
        "R3 [Ohm]":float(R[2]),
        "R4 [Ohm]":float(R[3])
    })

results=pd.DataFrame(results)
results

In [ ]:
from matplotlib.patches import Patch

fig=plt.figure(figsize=(10,7))
ax=fig.add_subplot(111,projection="3d")

SOC_mesh,T_mesh,_=make_r0_grid(models_ident[0])
R0_true=R0_true_function(SOC_mesh,T_mesh)

# R0 vrai
ax.plot_surface(
    np.array(SOC_mesh),
    np.array(T_mesh),
    np.array(R0_true),
    color="black",
    alpha=0.20,
    linewidth=0
)

CMAPS=[
    "viridis",
    "plasma",
    "cividis",
    "magma",
    "cool"
]

legend_handles=[
    Patch(
        facecolor="black",
        alpha=0.20,
        label="Vrai"
    )
]

# R0 issus des différentes initialisations
for i,model_i in enumerate(models_ident):

    _,_,R0_i=make_r0_grid(model_i)

    cmap_name=CMAPS[i%len(CMAPS)]
    cmap=plt.colormaps[cmap_name]

    ax.plot_surface(
        np.array(SOC_mesh),
        np.array(T_mesh),
        np.array(R0_i),
        cmap=cmap_name,
        alpha=0.30,
        linewidth=0
    )

    legend_handles.append(
        Patch(
            facecolor=cmap(0.6),
            alpha=0.5,
            label=f"Run {i+1}"
        )
    )

ax.set_xlabel("SOC")
ax.set_ylabel("Température [°C]")
ax.set_zlabel("R0 [Ohm]")
ax.set_title(
    "Sensibilité de R0 à l'initialisation — double shooting"
)

ax.legend(handles=legend_handles,loc="upper left")

plt.show()

On répète exactement la même identification avec différentes initialisations aléatoires. Si plusieurs runs donnent une loss similaire mais des paramètres ou des fonctions $R_0$ différents, cela met en évidence un problème potentiel d'identifiabilité.

# Note

Refaire le TP avec DOWNSAMPLE_STEP=1, remarques ?

# Conclusion

Ce TP montre qu'à partir de séries temporelles relativement simples — courant, tension et température — il est possible de reconstruire une partie de l'état interne d'une batterie et d'identifier les paramètres d'un modèle dynamique.

L'exercice met surtout en évidence plusieurs difficultés importantes :

- une bonne prédiction de la tension ne garantit pas que tous les paramètres identifiés soient physiquement corrects ;
- l'estimation du SOC par intégration peut dériver sur des fenêtres longues ;
- le choix de la fenêtre d'identification et des états initiaux influence fortement les résultats ;
- le multiple shooting permet de limiter une partie de ce drift en recalant certains états ;
- plusieurs jeux de paramètres peuvent parfois expliquer presque aussi bien les mêmes mesures : c'est la question de l'identifiabilité.

Le problème ne consiste donc pas uniquement à obtenir un faible écart sur la tension, mais aussi à vérifier que les états reconstruits et les paramètres estimés restent cohérents et robustes.

Enfin, ce type d'approche illustre l'intérêt des méthodes de **Scientific Machine Learning** : combiner un modèle physique dynamique, des paramètres classiques, un réseau de neurones et l'autodifférentiation dans un même problème d'optimisation.